# Fixed-tile MLP expansion: repeat and weight-layout check

Runs three rounds (seeds 0, 0, 1), with identical values in contiguous and transposed-weight layouts. Uses PyTorch with reduced-precision reductions allowed, and the fixed 64x64x64 / 4-warp Triton kernel. All four graphs rotate measurement order. Results are synthetic FP16 matmul timings, not full-model inference.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/fastinfer')
python = Path('/kaggle/working/fastinfer-t4-py312/bin/python')
assert python.exists(), 'Run the Python 3.12 / PyTorch 2.5.1 / Triton 3.1 setup first.'
assert (repo / 'benchmarks/matmul_compare.py').exists(), 'The fastinfer checkout is required.'

source = '"""Validate the single-row MLP win with identical values in two B layouts."""\n\nimport hashlib\nimport json\nimport runpy\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport torch\n\nfrom fastinfer.kernels.triton_matmul import matmul_kernel_resources, triton_matmul\n\n\nROOT = Path(__file__).resolve().parents[1]\nCONFIG = dict(block_m=64, block_n=64, block_k=64, num_warps=4)\n\n\ndef run_round(seed, round_index, helpers):\n    torch.manual_seed(seed)\n    a = torch.randn((1, 1536), device="cuda", dtype=torch.float16)\n    b = torch.randn((1536, 8960), device="cuda", dtype=torch.float16)\n    # Same values, but storage corresponds to contiguous linear weights W[N,K].\n    bt = b.t().contiguous().t()\n    assert torch.equal(b, bt)\n    expected = a.float() @ b.float()\n    operations = {}\n    layouts = {"contiguous": b, "weight_transpose": bt}\n    accuracy, resources = {}, {}\n    for layout, matrix in layouts.items():\n        operations[f"pytorch_{layout}"] = lambda x, y: x @ y\n        operations[f"triton_{layout}"] = lambda x, y: triton_matmul(x, y, **CONFIG)\n        resources[layout] = matmul_kernel_resources(a, matrix, **CONFIG)\n    inputs = {name: layouts[name.split("_", 1)[1]] for name in operations}\n    for name, op in operations.items():\n        result = op(a, inputs[name])\n        accuracy[name] = helpers["accuracy_against_fp32"](result, expected)\n        if name.startswith("triton"):\n            torch.testing.assert_close(result.float(), expected, rtol=.01, atol=.01)\n\n    stream = torch.cuda.Stream()\n    stream.wait_stream(torch.cuda.current_stream())\n    with torch.cuda.stream(stream):\n        for _ in range(10):\n            for name, op in operations.items():\n                op(a, inputs[name])\n    torch.cuda.current_stream().wait_stream(stream)\n    torch.cuda.synchronize()\n    graphs, outputs = {}, {}\n    names = list(operations)\n    # Vary graph construction order across repeats too.\n    shift = round_index % len(names)\n    for name in names[shift:] + names[:shift]:\n        graphs[name], outputs[name] = helpers["capture_graph"](\n            operations[name], a, inputs[name], 100)\n    for _ in range(10):\n        for graph in graphs.values():\n            graph.replay()\n    torch.cuda.synchronize()\n    samples = {name: [] for name in names}\n    for repetition in range(30):\n        shift = (repetition + round_index) % len(names)\n        for name in names[shift:] + names[:shift]:\n            start = torch.cuda.Event(enable_timing=True)\n            end = torch.cuda.Event(enable_timing=True)\n            start.record()\n            graphs[name].replay()\n            end.record()\n            end.synchronize()\n            samples[name].append(start.elapsed_time(end) * 1000 / 100)\n    result = {\n        "seed": seed, "round": round_index + 1,\n        "strides": {"a": list(a.stride()), **{\n            name: list(matrix.stride()) for name, matrix in layouts.items()}},\n        "accuracy": accuracy, "kernel_resources": resources,\n        "implementations": {name: helpers["summarize"](values)\n                            for name, values in samples.items()},\n    }\n    for layout in layouts:\n        pt, tt = (result["implementations"][f"{impl}_{layout}"]["latency_us"]["median"]\n                  for impl in ("pytorch", "triton"))\n        errors = [accuracy[f"{impl}_{layout}"]["mismatched_elements"]\n                  for impl in ("pytorch", "triton")]\n        print(f"Round {round_index + 1}, seed {seed}, {layout}: "\n              f"PyTorch {pt:.2f} us | Triton {tt:.2f} us | "\n              f"PT/Triton {pt / tt:.3f}x | mismatches {errors}", flush=True)\n    return result\n\n\ndef main():\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required")\n    torch.set_float32_matmul_precision("highest")\n    previous = torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True\n    helpers = runpy.run_path(str(ROOT / "benchmarks/matmul_compare.py"))\n    output = ROOT / "results/raw" / (\n        "mlp_expand_layout_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    report = {\n        "benchmark": "fixed_tile_mlp_expand_layout_check",\n        "environment": json.loads(subprocess.check_output(\n            [sys.executable, str(ROOT / "scripts/environment_report.py")],\n            cwd=ROOT, text=True)),\n        "script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "shape": {"m": 1, "k": 1536, "n": 8960}, "dtype": "float16",\n        "tile": CONFIG, "device": torch.cuda.current_device(),\n        "allow_fp16_reduced_precision_reduction": True,\n        "timing_mode": "cuda_graph_replay_amortized", "graph_ops": 100,\n        "warmup": 10, "graph_warmup": 10, "repetitions": 30,\n        "measurement_order": "rotating_four_implementations",\n        "correctness": {"reference": "pytorch_float32", "rtol": .01, "atol": .01},\n        "scope": "Synthetic FP16 matmul; transpose/copy preparation excluded from timing. "\n                 "No bias, actual model weights, or end-to-end inference.",\n        "rounds": [],\n    }\n    try:\n        with torch.inference_mode():\n            for index, seed in enumerate((0, 0, 1)):\n                report["rounds"].append(run_round(seed, index, helpers))\n                output.write_text(json.dumps(report, indent=2) + "\\n")\n                torch.cuda.empty_cache()\n    finally:\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = previous\n    print(f"Saved: {output}")\n\n\nif __name__ == "__main__":\n    main()\n'
script = repo / 'benchmarks/matmul_layout_check.py'
script.write_text(source)
subprocess.run([str(python), str(script)], cwd=repo, check=True)
